# Encoding cost: why routing and scheduling do not fit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/benchmarks/encoding-cost.ipynb)

Reproduces the analysis behind [logistics](https://zksf.org/applications/logistics/) and [manufacturing](https://zksf.org/applications/manufacturing/).

There is no quantum benchmark in this notebook, and **that is the result**. Both problems need more qubits to encode than any machine has or will soon have, so the comparison never gets as far as running.


In [ ]:
!pip install -q ortools numpy

## The arithmetic

A travelling salesman QUBO needs a binary variable per (stop, position) pair. Job-shop scheduling needs one per (job, machine, time slot). One qubit per variable, before any error correction.

In [ ]:
print(f"{'problem':<26}{'instance':<24}{'qubits needed':>15}")
for n in (5, 10, 20, 50, 100, 200):
    print(f"{'Vehicle routing (TSP)':<26}{f'{n} stops':<24}{n*n:>15,}")
for j, m, h in ((3,3,59), (5,5,147), (10,10,570), (15,15,1193)):
    print(f"{'Job-shop scheduling':<26}{f'{j}x{m}, {h} slots':<24}{j*m*h:>15,}")
print("\nLargest QPU on the ZKSF platform today: 108 physical qubits.")

## What classical does with the same instances

OR-Tools, on an ordinary CPU. Compare the times against the qubit counts above.

In [ ]:
import time, numpy as np
from ortools.constraint_solver import routing_enums_pb2, pywrapcp

for n in (20, 100, 200):
    rng = np.random.default_rng(20260902 + n)
    pts = rng.uniform(0, 1000, (n, 2))
    D = np.round(np.linalg.norm(pts[:,None]-pts[None,:], axis=2)).astype(int)
    mgr = pywrapcp.RoutingIndexManager(n, 1, 0); routing = pywrapcp.RoutingModel(mgr)
    cb = routing.RegisterTransitCallback(lambda i, j: int(D[mgr.IndexToNode(i)][mgr.IndexToNode(j)]))
    routing.SetArcCostEvaluatorOfAllVehicles(cb)
    prm = pywrapcp.DefaultRoutingSearchParameters()
    prm.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    prm.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    prm.time_limit.FromSeconds(3)
    t0 = time.perf_counter(); sol = routing.SolveWithParameters(prm)
    print(f"routing {n:>3} stops -> tour {sol.ObjectiveValue():>7,} "
          f"({time.perf_counter()-t0:.1f}s budget)   quantum needs {n*n:,} qubits")

In [ ]:
from ortools.sat.python import cp_model

for jobs, machines in ((5,5), (10,10), (15,15)):
    rng = np.random.default_rng(20260902 + jobs)
    dur = rng.integers(2, 10, (jobs, machines))
    order = [list(rng.permutation(machines)) for _ in range(jobs)]
    hz = int(dur.sum()); m = cp_model.CpModel()
    starts, ends, ivs = {}, {}, {k: [] for k in range(machines)}
    for j in range(jobs):
        for pos, mm in enumerate(order[j]):
            s_ = m.NewIntVar(0, hz, ""); e_ = m.NewIntVar(0, hz, "")
            ivs[mm].append(m.NewIntervalVar(s_, int(dur[j][mm]), e_, ""))
            starts[j,mm], ends[j,mm] = s_, e_
            if pos: m.Add(s_ >= ends[j, order[j][pos-1]])
    for mm in range(machines): m.AddNoOverlap(ivs[mm])
    mk = m.NewIntVar(0, hz, "mk")
    m.AddMaxEquality(mk, [ends[j, order[j][-1]] for j in range(jobs)]); m.Minimize(mk)
    sv = cp_model.CpSolver(); sv.parameters.max_time_in_seconds = 10.0
    t0 = time.perf_counter(); st = sv.Solve(m); dt = time.perf_counter() - t0
    print(f"job-shop {jobs}x{machines} -> makespan {int(sv.ObjectiveValue()):>4} in {dt:.2f}s "
          f"({'PROVEN OPTIMAL' if st == cp_model.OPTIMAL else 'feasible'}), "
          f"quantum needs {jobs*machines*hz:,} qubits")

## Read the two columns together

At industrial sizes the classical solver answers in seconds, and for job-shop it **proves** the answer is optimal. The quantum encoding of the same instance needs tens or hundreds of thousands of qubits before error correction.

The gap is not one hardware generation. It is four orders of magnitude against a method that already finishes faster than you can read the output.

## What would have to change

Not hardware first. **The encoding.** Quadratic and time-indexed scaling are properties of the standard QUBO formulations, not laws. Until a formulation reduces the cost by orders of magnitude, better qubits do not help, because you cannot reach 40,000 logical qubits by improving fidelity.

If a vendor offers quantum routing or scheduling, the question is not how many qubits their machine has. It is how many binary variables their encoding needs at the size you actually run.

## Next

- [The full benchmark page](https://zksf.org/applications/logistics/), with the analysis and the caveats
- [How we benchmark](https://zksf.org/applications/methodology/): the rules every one of these follows
- [All applications](https://zksf.org/applications/) across six sectors
- [Certification](https://zksf.org/quantum-computing-certification/): what the accuracy statements assert
